# EDA all_stocks_train.csv

Phan tich tong quan, NaN, direction_5d, win rate theo thang/nganh, top ma va tuong quan feature.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', 200)

df = pd.read_csv('all_stocks_train.csv', parse_dates=['trading_date'])
df = df.sort_values(['symbol', 'trading_date']).reset_index(drop=True)
df['month'] = df['trading_date'].dt.to_period('M').astype(str)
df['direction_5d_label'] = df['direction_5d'].map({1: 'Tang', 0: 'Giam'})
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [
    c for c in numeric_cols
    if c not in ['open', 'high', 'low', 'close', 'volume', 'direction_5d', 'direction_10d', 'return_5d']
]

In [ ]:
overview = {
    'so_ma': df['symbol'].nunique(),
    'so_dong': len(df),
    'tu_ngay': df['trading_date'].min(),
    'den_ngay': df['trading_date'].max(),
}
overview

In [ ]:
nan_ratio = (df.isna().mean() * 100).sort_values(ascending=False).to_frame('nan_pct')
display(nan_ratio)

plt.figure(figsize=(18, 8))
sns.heatmap(df.isna().T, cbar=False, cmap='mako')
plt.title('Heatmap NaN theo cot')
plt.xlabel('Dong du lieu')
plt.ylabel('Cot')
plt.tight_layout()
plt.show()

In [ ]:
direction_dist = (
    df['direction_5d']
    .dropna()
    .map({1: 'Tang', 0: 'Giam'})
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
display(direction_dist)

monthly_win = (
    df.dropna(subset=['direction_5d'])
      .groupby('month')['direction_5d']
      .mean()
      .mul(100)
      .reset_index(name='win_rate_pct')
)
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_win, x='month', y='win_rate_pct', marker='o')
plt.xticks(rotation=60)
plt.title('Win rate direction_5d theo thang')
plt.ylabel('Win rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
industry_win = (
    df.dropna(subset=['direction_5d'])
      .groupby('industry')
      .agg(win_rate=('direction_5d', 'mean'), rows=('symbol', 'size'))
      .query('rows >= 50')
      .sort_values('win_rate', ascending=False)
      .reset_index()
)
plt.figure(figsize=(14, 8))
sns.barplot(data=industry_win.head(20), x='win_rate', y='industry', palette='crest')
plt.title('Top nganh theo win rate direction_5d')
plt.xlabel('Win rate')
plt.tight_layout()
plt.show()

top_symbols = (
    df.dropna(subset=['direction_5d'])
      .groupby('symbol')
      .agg(win_rate=('direction_5d', 'mean'), days=('trading_date', 'count'))
      .query('days >= 100')
      .sort_values('win_rate', ascending=False)
      .head(10)
)
display(top_symbols)

In [ ]:
corr = df[feature_cols + ['direction_5d']].corr(numeric_only=True)['direction_5d'].drop('direction_5d').sort_values(ascending=False)
display(corr.to_frame('corr_with_direction_5d'))

plt.figure(figsize=(10, 12))
corr.sort_values().plot(kind='barh', color='teal')
plt.title('Tuong quan giua feature va direction_5d')
plt.tight_layout()
plt.show()

In [ ]:
nan_ok = nan_ratio['nan_pct'].fillna(100).lt(20).mean() >= 0.7
label_balance = direction_dist.max() <= 65 if not direction_dist.empty else False
enough_symbols = overview['so_ma'] >= 30
enough_rows = overview['so_dong'] >= 10000

print('Ket luan so bo:')
print(f'- Du lieu da dang ma co phieu: {enough_symbols}')
print(f'- So dong du lieu du lon: {enough_rows}')
print(f'- Da so cot co NaN chap nhan duoc: {nan_ok}')
print(f'- Label direction_5d khong lech qua manh: {label_balance}')

if all([nan_ok, label_balance, enough_symbols, enough_rows]):
    print('\n=> Data nhin chung du tot de train, nhung van can xu ly NaN va kiem tra leakage truoc khi vao model.')
else:
    print('\n=> Data chua san sang hoan toan. Can xu ly bo sung NaN, can bang label hoac mo rong lich su truoc khi train.')